# 48 - Open Benchmarks And Evaluation Harnesses

## What / Why / How

**What we are trying to do:** Learn how open benchmarks and evaluation harnesses make robotics results more reproducible.

**Why this matters:** Training a policy is easier than proving it works. Open evaluation is becoming a major part of trustworthy robotics.

**How we will do it:** Create a benchmark matrix, compute confidence intervals, and rank models while accounting for task mismatch and uncertainty.

## Evaluation Harnesses

Open evaluation is now as important as open training. A model that wins one benchmark may fail another because sensors, action spaces, objects, instructions, or time horizons differ.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
benchmarks = [
    {"name": "LIBERO", "focus": "lifelong manipulation", "horizon": "short/medium", "visual": True},
    {"name": "CALVIN", "focus": "long-horizon language tasks", "horizon": "long", "visual": True},
    {"name": "ManiSkill", "focus": "GPU manipulation sim/data", "horizon": "short/medium", "visual": True},
    {"name": "RoboCasa365", "focus": "household mobile manipulation", "horizon": "medium/long", "visual": True},
    {"name": "RoboTwin 2.0", "focus": "bimanual data generation", "horizon": "medium", "visual": True},
    {"name": "SimplerEnv", "focus": "real-to-sim policy eval", "horizon": "short", "visual": True},
]
for b in benchmarks:
    print(f"{b['name']:12} {b['focus']:34} horizon={b['horizon']}")

In [ ]:
rng = np.random.default_rng(48)
models = ["SmolVLA", "OpenVLA", "pi0.5", "Xiaomi-Robotics-0"]
true_success = {"SmolVLA": 0.62, "OpenVLA": 0.66, "pi0.5": 0.72, "Xiaomi-Robotics-0": 0.75}
episodes = 120
results = {}
for model in models:
    successes = rng.binomial(episodes, true_success[model])
    p = successes / episodes
    se = math.sqrt(p * (1 - p) / episodes)
    results[model] = (p, 1.96 * se)
for model, (p, ci) in sorted(results.items(), key=lambda x: -x[1][0]):
    print(f"{model:18} success={p:.3f}  95% CI +/- {ci:.3f}")

In [ ]:
# Penalize models evaluated on mismatched benchmarks.
task_match = {"SmolVLA": 0.8, "OpenVLA": 0.75, "pi0.5": 0.7, "Xiaomi-Robotics-0": 0.65}
reproducibility = {"SmolVLA": 0.85, "OpenVLA": 0.8, "pi0.5": 0.75, "Xiaomi-Robotics-0": 0.65}
for model in models:
    p, ci = results[model]
    adjusted = p * task_match[model] * reproducibility[model]
    print(f"{model:18} raw={p:.3f} adjusted_for_your_lab={adjusted:.3f}")

## Exercises

1. Increase episodes from 120 to 1000 and inspect confidence intervals.
2. Add a real-robot evaluation column.
3. Explain why leaderboard rankings should not directly choose your capstone model.